In [7]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

In [ ]:
from dotenv import load_dotenv
load_dotenv()
from google import genai
gemini_client = genai.Client()

In [9]:
documents = load_faq_data()
index = build_index(documents)

In [10]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=gemini_client,
    instructions=instructions,
)

In [11]:
answer = assistant.rag('How do I run Ollama locally?')
print(answer)

To run Ollama locally, follow these steps:

1.  **Install Ollama:** Visit [https://ollama.com/download](https://ollama.com/download) and download the version for your operating system:
    *   **macOS**: Download and install the `.pkg`.
    *   **Windows**: Download and install the `.msi`.
    *   **Linux**: Run this command in your terminal: `curl -fsSL https://ollama.com/install.sh | sh`.

2.  **Start the model:** Once installed, open a terminal and run:
    ```bash
    ollama run llama3
    ```
    This will download the LLaMA 3 model and open a chat-like interface.

3.  **Test the server:** You can verify the local server is running by executing:
    ```bash
    curl http://localhost:11434
    ```

4.  **Use with Python (Optional):** You can install the Python client using `pip install ollama` to interact with the model through code.


In [12]:
answer = assistant.rag('How do I run Olama locally?')
print(answer)

The provided context does not contain information on how to run Ollama locally.


In [13]:
messages = [{'role': 'user', 'parts': [{'text': 'I just discovered the course. Can I join it?'}]}]

response = gemini_client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=messages,
)

response.text

'To give you an accurate answer, I need a little more information, as I am an AI and don\'t know which specific course you are referring to.\n\n**Could you please clarify:**\n\n1.  **What is the name of the course?**\n2.  **Where did you find it?** (e.g., Coursera, Udemy, a university website, a company training portal, or a social media post?)\n\n---\n\n### In the meantime, here is how you can usually find out:\n\n*   **Check the Platform/Website:** Look for an "Enroll," "Join," or "Sign Up" button on the course landing page. If the button is greyed out or says "Closed," the registration period may have ended.\n*   **Check for "Self-Paced" vs. "Cohort-Based":**\n    *   **Self-Paced:** These courses almost always allow you to join at any time.\n    *   **Cohort-Based:** These courses follow a specific schedule with a start and end date. If the start date has passed, you might be too late, though some instructors allow late enrollment.\n*   **Look for a Contact Email:** If you aren\'t 

In [14]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [15]:
from google.genai import types

search_function = types.FunctionDeclaration(
    name='search',
    description='Search the FAQ database for entries matching the given query.',
    parameters={
        'type': 'object',
        'properties': {
            'query': {
                'type': 'string',
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        'required': ['query']
    }
)

search_tool = types.Tool(function_declarations=[search_function])

In [16]:
response = gemini_client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=messages,
    config={"tools": [search_tool]}
)

response.function_calls


[FunctionCall(
   args={
     'query': 'can I join the course late'
   },
   id='b3kVLZoF',
   name='search'
 )]

In [17]:
len(response.function_calls)

1

In [18]:
call = response.function_calls[0]

In [19]:
call

FunctionCall(
  args={
    'query': 'can I join the course late'
  },
  id='b3kVLZoF',
  name='search'
)

In [20]:
args = call.args
args

{'query': 'can I join the course late'}

In [21]:
call.name

'search'

In [22]:
results = search(**args)


In [23]:
function_response_part = types.Part(
    function_response=types.FunctionResponse(
        id=call.id,
        name=call.name,
        response={'result': results},
    )
)

In [24]:

messages.append(response.candidates[0].content)

messages.append(types.Content(role='user', parts=[function_response_part]))

In [25]:
messages

[{'role': 'user',
  'parts': [{'text': 'I just discovered the course. Can I join it?'}]},
 Content(
   parts=[
     Part(
       function_call=FunctionCall(
         args={
           'query': 'can I join the course late'
         },
         id='b3kVLZoF',
         name='search'
       ),
       thought_signature=b'\x124\n2\x01\x11M2\x0fB\x9bX\x85w\rK\x9e\\\x92\xe4\x11\xdb\xfc\xbf\x86\x80\xc7R\xa4\x10VD\xf5\xaa\xedH\x17\xeb\x14\n&\x13\xe1\xcd\xbd(\x80\xacUG\x82]m-'
     ),
   ],
   role='model'
 ),
 Content(
   parts=[
     Part(
       function_response=FunctionResponse(
         id='b3kVLZoF',
         name='search',
         response={
           'result': [
             {<... 5 items at Max depth ...>},
             {<... 5 items at Max depth ...>},
             {<... 5 items at Max depth ...>},
             {<... 5 items at Max depth ...>},
             {<... 5 items at Max depth ...>},
           ]
         }
       )
     ),
   ],
   role='user'
 )]

In [26]:
response = gemini_client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=messages,
    config={"tools": [search_tool]}
)



In [27]:
print(response.text)


Yes, you can join the course at any time.

However, please note the following regarding certificates and participation:

*   **Certificates:** If you wish to receive a certificate, you must submit your project while submissions are still being accepted. Note that certificates are only awarded for finishing the course with a "live" cohort; they are not awarded for self-paced mode because you need to participate in the peer-review process, which is only available while the course is running.
*   **Getting Started:** You don't need to formally register to start learning. You can begin watching videos, working through the notebooks, and submitting homework (provided the submission form is still open) by following the materials available on the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp). 
*   **Workflow:** You can find deadlines and submit your homework through the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).


In [28]:
def calculate_gemini_cost(usage_metadata, input_price_per_million, output_price_per_million):
    prompt_tokens = getattr(usage_metadata, "prompt_token_count", 0) or 0
    candidate_tokens = getattr(usage_metadata, "candidates_token_count", 0) or 0
    thoughts_tokens = getattr(usage_metadata, "thoughts_token_count", 0) or 0

    billed_input_tokens = prompt_tokens
    billed_output_tokens = candidate_tokens + thoughts_tokens

    input_cost = billed_input_tokens / 1_000_000 * input_price_per_million
    output_cost = billed_output_tokens / 1_000_000 * output_price_per_million
    total_cost = input_cost + output_cost

    return {
        'prompt_tokens': billed_input_tokens,
        'output_tokens': billed_output_tokens,
        'input_cost': input_cost,
        'output_cost': output_cost,
        'total_cost': total_cost,
    }

pricing_by_model = {
    'gemini-3.1-flash-lite': {'input': 0.30, 'output': 2.50},
}

usage_metadata = response.usage_metadata
model_name = getattr(response, 'model_version', 'gemini-3.1-flash-lite')
pricing = pricing_by_model.get(model_name, pricing_by_model['gemini-3.1-flash-lite'])

cost_breakdown = calculate_gemini_cost(
    usage_metadata,
    pricing['input'],
    pricing['output'],
)

print(f"Model: {model_name}")
print(f"Prompt tokens: {cost_breakdown['prompt_tokens']}")
print(f"Output tokens: {cost_breakdown['output_tokens']}")
print(f"Estimated input cost: ${cost_breakdown['input_cost']:.6f}")
print(f"Estimated output cost: ${cost_breakdown['output_cost']:.6f}")
print(f"Estimated total cost: ${cost_breakdown['total_cost']:.6f}")


Model: gemini-3.1-flash-lite
Prompt tokens: 822
Output tokens: 215
Estimated input cost: $0.000247
Estimated output cost: $0.000538
Estimated total cost: $0.000784


In [29]:
from google.genai import types

def make_call(call):
    # Gemini natively parses args into a dictionary, so json.loads is not needed
    args = call.args

    if call.name == 'search':
        result = search(**args)

    # Gemini expects a specific FunctionResponse object rather than a raw JSON string/dictionary
    return types.Part(
        function_response=types.FunctionResponse(
            id=call.id,
            name=call.name,
            response={'result': result},
        )
    )

In [30]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""
question = 'I just discovered the course. Can I join it?'
config = types.GenerateContentConfig(
    system_instruction=instructions,
    tools=[search_tool]
)

messages = [
    {'role': 'user', 'parts': [{'text': question}]}
]

In [31]:
response = gemini_client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=messages,
    config=config
)


In [32]:
messages.append(response.candidates[0].content)

function_responses = []

if response.function_calls:
    for call in response.function_calls:
        print('function_call:', call.name, call.args)
        
        # Execute the function using your make_call method
        call_output = make_call(call)
        function_responses.append(call_output)
        
    # Append the results of the functions back to the conversation as a single user message
    messages.append(types.Content(role='user', parts=function_responses))
else:
    # If no functions were called, print the text response
    print('ASSISTANT:')
    print(response.text)

function_call: search {'query': 'can I join the course late registration enrollment policy'}


In [33]:
messages 

[{'role': 'user',
  'parts': [{'text': 'I just discovered the course. Can I join it?'}]},
 Content(
   parts=[
     Part(
       function_call=FunctionCall(
         args={
           'query': 'can I join the course late registration enrollment policy'
         },
         id='gcUAc5TN',
         name='search'
       ),
       thought_signature=b'\x124\n2\x01\x11M2\x0f\x0b^\xd3\xef\x12b\x01\xf0\x93\xa9\x85\xe4y\xf4\x8d\xc9x\xb2\xebww)\xec\xdc}W\xe1qZ\x1a\xb1n\x15d$\xdbQ\xf7\xa3\xfd\xa6\xe4\xc6`_'
     ),
   ],
   role='model'
 ),
 Content(
   parts=[
     Part(
       function_response=FunctionResponse(
         id='gcUAc5TN',
         name='search',
         response={
           'result': [
             {<... 5 items at Max depth ...>},
             {<... 5 items at Max depth ...>},
             {<... 5 items at Max depth ...>},
             {<... 5 items at Max depth ...>},
             {<... 5 items at Max depth ...>},
           ]
         }
       )
     ),
   ],
   role='user'
 

In [34]:
messages = [
    {'role': 'user', 'parts': [{'text': question}]}
]

it = 1

while True:
    print(f'iteration #{it}...')
    has_function_calls = False

    response = gemini_client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=messages,
        config=config # Uses the config we defined in the previous setup cell
    )

    if response.function_calls:
        has_function_calls = True
        
        # Append the model's function call request
        messages.append(response.candidates[0].content)
        
        function_responses = []
        
        # Loop through and execute each requested function
        for call in response.function_calls:
            print('function_call:', call.name, call.args)
            
            # Use the make_call function we converted earlier
            call_output = make_call(call)
            function_responses.append(call_output)
            
        # Append all function outputs as a single user message part
        messages.append(types.Content(role='user', parts=function_responses))
        
    else:
        # If no tools were called, print the final answer
        print('ASSISTANT:')
        print(response.text)
    
    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
function_call: search {'query': 'join the course enrollment policy'}
iteration #2...
ASSISTANT:
Yes, you can absolutely join! You don't need to formally register to get started—you can simply begin working through the materials.

Here is the important information you should know:

*   **Materials:** All the videos, notebooks, and GitHub materials are available for you to use. You can start whenever you like.
*   **Homework & Projects:** If you wish to receive a certificate, you need to submit your project and complete the required peer reviews while the course is still accepting submissions. Please note that certificates are only awarded when following along with a "live" cohort; they are not available for self-paced study.
*   **How to Start:**
    1.  Check the [LLM Zoomcamp documentation](https://datatalks.club/docs/courses/llm-zoomcamp/).
    2.  Review the [general Zoomcamp logistics](https://datatalks.club/docs/courses/zoomcamp-logistics/).
    3.  Follow along wi

In [35]:
import time
def agent_loop(instructions, question, model="gemini-3.1-flash-lite") -> str:
    # Initialize the config with system instructions inside the function
    config = types.GenerateContentConfig(
        system_instruction=instructions,
        tools=[search_tool]
    )
    
    messages = [
        {'role': 'user', 'parts': [{'text': question}]}
    ]

    it = 1

    while True:
        time.sleep(4.5) # added the time.sleep to avoid rate limiting
        print(f'iteration #{it}...')
        has_function_calls = False

        response = gemini_client.models.generate_content(
            model=model,
            contents=messages,
            config=config
        )

        if response.function_calls:
            has_function_calls = True
            messages.append(response.candidates[0].content)
            
            function_responses = []
            
            for call in response.function_calls:
                print('function_call:', call.name, call.args)
                call_output = make_call(call)
                function_responses.append(call_output)
                
            messages.append(types.Content(role='user', parts=function_responses))
            
        else:
            last_answer = response.text
            print('ASSISTANT:')
            print(last_answer)
            
        it = it + 1
        if has_function_calls == False:
            break
            
    return last_answer

In [36]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'

In [37]:
result = agent_loop(instructions, question)

iteration #1...
function_call: search {'query': 'can I join the course late registration enrollment'}
iteration #2...


ASSISTANT:
Yes, you are absolutely welcome to join! You don't need a formal registration to get started—you can simply begin learning and working through the materials.

Here are a few key things to keep in mind:

*   **How to start:** You can jump right in by watching the videos, working through the notebooks, and following the instructions on the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).
*   **Homework:** You can still submit homework as long as the submission forms are open.
*   **Certificates:** Please note that if you are interested in receiving a certificate, you must be able to submit your final project and participate in the peer-review process while the course is running. The course does not award certificates for self-paced learning, as the peer-review process requires a live cohort.

For further details on how to follow the workflow, you can check out the [LLM Zoomcamp documentation](https://datatalks.club/docs/courses/llm-zoomcamp/).



In [38]:
result

"Yes, you are absolutely welcome to join! You don't need a formal registration to get started—you can simply begin learning and working through the materials.\n\nHere are a few key things to keep in mind:\n\n*   **How to start:** You can jump right in by watching the videos, working through the notebooks, and following the instructions on the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n*   **Homework:** You can still submit homework as long as the submission forms are open.\n*   **Certificates:** Please note that if you are interested in receiving a certificate, you must be able to submit your final project and participate in the peer-review process while the course is running. The course does not award certificates for self-paced learning, as the peer-review process requires a live cohort.\n\nFor further details on how to follow the workflow, you can check out the [LLM Zoomcamp documentation](https://datatalks.club/docs/courses/llm-zoomcamp/).\n\n

In [39]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
"""

question = "what's queen gambit?"

result = agent_loop(instructions, question)

iteration #1...
function_call: search {'query': "what's queen gambit?"}
iteration #2...
ASSISTANT:
I have searched the course FAQ, and there is no information regarding "Queen's Gambit." This appears to be off-topic to the course material.

Are there any other areas related to the course that you would like to explore?


In [40]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {'query': 'run Olama locally'}
iteration #2...
function_call: search {'query': 'Ollama'}
iteration #3...
ASSISTANT:
To run Ollama locally, follow these installation steps based on your operating system:

1.  **Installation:**
    *   **macOS**: Download the `.pkg` and install it from [ollama.com/download](https://ollama.com/download).
    *   **Windows**: Download the `.msi` and install it from [ollama.com/download](https://ollama.com/download).
    *   **Linux**: Run the following command in your terminal:
        ```bash
        curl -fsSL https://ollama.com/install.sh | sh
        ```

2.  **Starting a Model:**
    Once installed, open a terminal and type the following to download and start a model (e.g., LLaMA 3):
    ```bash
    ollama run llama3
    ```
    This will open a chat-like interface directly in your terminal.

3.  **Testing the Server:**
    To confirm the local Ollama server is running, you can use:
    ```bash
    curl http://loc

'To run Ollama locally, follow these installation steps based on your operating system:\n\n1.  **Installation:**\n    *   **macOS**: Download the `.pkg` and install it from [ollama.com/download](https://ollama.com/download).\n    *   **Windows**: Download the `.msi` and install it from [ollama.com/download](https://ollama.com/download).\n    *   **Linux**: Run the following command in your terminal:\n        ```bash\n        curl -fsSL https://ollama.com/install.sh | sh\n        ```\n\n2.  **Starting a Model:**\n    Once installed, open a terminal and type the following to download and start a model (e.g., LLaMA 3):\n    ```bash\n    ollama run llama3\n    ```\n    This will open a chat-like interface directly in your terminal.\n\n3.  **Testing the Server:**\n    To confirm the local Ollama server is running, you can use:\n    ```bash\n    curl http://localhost:11434\n    ```\n    If successful, you will receive a JSON response containing the models.\n\n4.  **Using in Python:**\n    Yo

In [41]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {'query': 'queen gambit'}
iteration #2...
ASSISTANT:
I've checked the course FAQ, and there is no information about "Queen's Gambit." Since my role is to assist with questions related specifically to this course and its logistics, I am unable to provide information on this topic.

Are there any other areas of the course you would like to explore?


'I\'ve checked the course FAQ, and there is no information about "Queen\'s Gambit." Since my role is to assist with questions related specifically to this course and its logistics, I am unable to provide information on this topic.\n\nAre there any other areas of the course you would like to explore?'

In [42]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")
# ALL good till here

iteration #1...
function_call: search {'query': 'queen gambit'}
iteration #2...
ASSISTANT:
I'm sorry, but I couldn't find any information about "queen gambit" in the course FAQ. As a teaching assistant, I can only answer questions related to this course and its logistics.

Are there any other areas of the course you would like to explore?


'I\'m sorry, but I couldn\'t find any information about "queen gambit" in the course FAQ. As a teaching assistant, I can only answer questions related to this course and its logistics.\n\nAre there any other areas of the course you would like to explore?'

In [43]:

# Cleanup note
# We are not reproducing the workshop's toyaikit runner layer here.
# That version adds an extra abstraction on top of the direct Gemini API,
# depends on package-specific runner classes, and is beyond the minimal scope
# of this notebook.
# Instead, we keep the manual Gemini tool-calling flow above, which shows the
# same idea with less complexity and no extra dependency on external runner files.